# KhadSan AI รอบ 2 : ทำให้โมเดลเสถียรขึ้น

รอบ 1 เจอปัญหา 3 อย่าง รอบนี้แก้ทั้งหมด โดยทุกการทดลองใช้ train/val/test **ชุดเดียวกัน** เทียบกันได้ยุติธรรม

| ปัญหาจากรอบ 1 | การทดลองที่แก้ |
|---|---|
| รูปจากแหล่งอื่น (Kaggle) ทายถูกแค่ 58.8% | `8_cosine_more_data` = ค่าเดียวกับตัวดีที่สุด แต่เพิ่มรูป Kaggle 3,000 รูปเข้าชุดเทรน |
| ตีกรอบหน้าคนเป็นผลไม้ | `8_cosine_more_data` เพิ่มรูปพื้นหลังที่ไม่มีผลไม้ 126 รูป (COCO128) ด้วย |
| RMSProp loss ระเบิด คะแนนแกว่ง | `9_rmsprop_low_lr` = RMSProp ลด LR จาก 0.001 เป็น 0.0001 |

ตัวเทียบ (เทรนซ้ำบนข้อมูลชุดเดียวกัน) : `5_cosine_lr` (ข้อมูลเดิม) และ `4_rmsprop` (LR 0.001)

**ก่อนกด Run all**
1. push โฟลเดอร์ `train/` ล่าสุดขึ้น GitHub
2. Runtime > Change runtime type > **T4 GPU**
3. ใน Drive สร้างโฟลเดอร์ `KhadSan_detect_v2` แล้วอัปโหลด 2 ไฟล์ :
   - `KaggleBoxes.zip` (รูป Kaggle ที่ตีกรอบแล้ว + รูปพื้นหลัง สร้างจาก `python prepare_data.py --make-boxes`)
   - `ExternalTest.zip` (ชุดทดสอบภายนอก 600 รูป)
4. dataset หลัก : ตั้ง Colab Secret `ROBOFLOW_API_KEY` **หรือ** อัปโหลด `FruitsQuality.zip` ไว้ในโฟลเดอร์เดียวกัน

ใช้เวลาราว 2.5-3 ชั่วโมง ถ้าหลุดกลางทาง กด Run all ใหม่ การทดลองที่เสร็จแล้วจะถูกข้าม

## 1) เช็กว่าได้ GPU
ถ้าขึ้นว่าไม่เจอ GPU : Runtime > Change runtime type > **T4 GPU** > Save แล้วกด Run all ใหม่

In [ ]:
import torch
if not torch.cuda.is_available():
    raise Exception("ไม่เจอ GPU ไปที่ Runtime > Change runtime type > เลือก T4 GPU แล้วกด Run all ใหม่")
print("ได้ GPU:", torch.cuda.get_device_name(0))

## 2) ติดตั้งไลบรารี
ล็อกรุ่น ultralytics ให้ตรงกับที่ทีมใช้ (8.4.191) ผลจะได้เทียบกันได้

In [ ]:
!pip install -q ultralytics==8.4.191 roboflow onnx onnxruntime
!yolo settings sync=False
import ultralytics
print("ultralytics", ultralytics.__version__)
if ultralytics.__version__ != "8.4.191":
    raise Exception("รุ่น ultralytics ไม่ตรง ให้ Runtime > Restart session แล้วกด Run all ใหม่")

## 3) ดึงโค้ดเทรนจาก GitHub ของทีม
ต้อง **push โค้ดล่าสุดขึ้น GitHub ก่อน** ช่องนี้จะเช็กให้ว่าโค้ดที่ดึงมาเป็นรุ่นที่ใช้กับรอบนี้ได้

In [ ]:
import os
REPO = "https://github.com/srirat-intalai/khadsan-ai"

# ดึงครั้งแรก = clone / รันซ้ำ = อัปเดตเป็นโค้ดล่าสุด
if os.path.isdir("/content/khadsan-ai"):
    !git -C /content/khadsan-ai pull -q
else:
    !git clone -q {REPO} /content/khadsan-ai

CODE = "/content/khadsan-ai/train"
!git -C /content/khadsan-ai log -1 --format="โค้ดรุ่นล่าสุด: %h %s (%ad)" --date=short

# เช็กว่าโค้ดมีของที่รอบนี้ต้องใช้ (ถ้าไม่มี แปลว่ายังไม่ได้ push โค้ดล่าสุด)
train_code = open(CODE + "/train_yolo.py", encoding="utf-8").read()
prep_code = open(CODE + "/prepare_data.py", encoding="utf-8").read()

for need in ["8_cosine_more_data", "9_rmsprop_low_lr", "--more-data", "external_"]:
    if need not in train_code:
        raise Exception("train_yolo.py ใน GitHub ยังเป็นรุ่นเก่า (ไม่มี " + need + ") ให้ push โค้ดล่าสุดก่อน")
for need in ["--extra", "data_more.yaml"]:
    if need not in prep_code:
        raise Exception("prepare_data.py ใน GitHub ยังเป็นรุ่นเก่า (ไม่มี " + need + ") ให้ push โค้ดล่าสุดก่อน")

print("โค้ดครบ พร้อมเทรน")

## 4) ต่อ Google Drive แล้วโหลดข้อมูลทั้ง 3 ชุด
ผลเก็บใน Drive โฟลเดอร์ `KhadSan_detect_v2` (ไม่ทับรอบ 1)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/KhadSan_detect_v2"
os.makedirs(OUT, exist_ok=True)
print("ผลจะเก็บที่", OUT)

# ----- dataset หลัก (Fruits Quality) : ใช้ zip ใน Drive ถ้ามี ไม่มีก็โหลดจาก Roboflow ด้วย API key -----
SRC = "/content/FruitsQuality"
if os.path.exists(OUT + "/FruitsQuality.zip"):
    print("ใช้ FruitsQuality.zip จาก Drive")
    !unzip -q -o "{OUT}/FruitsQuality.zip" -d /content/zip_tmp
    for root, dirs, files in os.walk("/content/zip_tmp"):
        if "data.yaml" in files:
            SRC = root
            break
else:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    project = rf.workspace("alex-day-qzfwa").project("fruits-quality-qtdtf")
    project.version(1).download("yolov8", location=SRC)

# เช็ก dataset หลัก : ต้องมี data.yaml และรูปรวมกันประมาณ 2,583 รูป
if not os.path.exists(SRC + "/data.yaml"):
    raise Exception("ไม่เจอ data.yaml ใน dataset หลัก เช็กไฟล์ zip หรือ API key")
count = 0
for part in ["train", "valid", "test"]:
    if os.path.isdir(SRC + "/" + part + "/images"):
        count = count + len(os.listdir(SRC + "/" + part + "/images"))
print("dataset หลัก", count, "รูป (ควรได้ 2583)")
if count != 2583:
    print("!!! จำนวนรูปไม่ตรงกับที่ทีมใช้ ผลอาจต่างจากเดิม")

# ----- ชุดทดสอบภายนอก (ExternalTest.zip) : 6 คลาส คลาสละ 100 รูป -----
EXTERNAL = ""
if os.path.exists(OUT + "/ExternalTest.zip"):
    !unzip -q -o "{OUT}/ExternalTest.zip" -d /content/external_tmp
    for root, dirs, files in os.walk("/content/external_tmp"):
        if "Apple_Good" in dirs:
            EXTERNAL = root
            break
    total = 0
    for c in sorted(os.listdir(EXTERNAL)):
        if os.path.isdir(EXTERNAL + "/" + c):
            n = len(os.listdir(EXTERNAL + "/" + c))
            total = total + n
            print("  ", c, n, "รูป")
    print("ชุดทดสอบภายนอก", total, "รูป (ควรได้ 600)")
else:
    print("ไม่มี ExternalTest.zip ข้ามการทดสอบภายนอก (แนะนำให้อัปโหลด)")

if EXTERNAL == "":
    raise Exception("รอบ 2 ต้องมี ExternalTest.zip ใน " + OUT + " (ใช้วัดว่ารูปจากแหล่งอื่นแม่นขึ้นไหม)")

# ----- รูป Kaggle ที่ตีกรอบแล้ว + รูปพื้นหลัง (KaggleBoxes.zip) -----
if not os.path.exists(OUT + "/KaggleBoxes.zip"):
    raise Exception("รอบ 2 ต้องอัปโหลด KaggleBoxes.zip ไว้ใน " + OUT + " ก่อน")
!unzip -q -o "{OUT}/KaggleBoxes.zip" -d /content/kaggle_tmp
EXTRA = ""
for root, dirs, files in os.walk("/content/kaggle_tmp"):
    if "images" in dirs and "labels" in dirs:
        EXTRA = root
        break

# เช็กให้ครบ : รูปกับไฟล์ป้ายต้องจับคู่กันครบทุกรูป
images = os.listdir(EXTRA + "/images")
labels = os.listdir(EXTRA + "/labels")
fruit = 0
background = 0
for f in images:
    if f.startswith("background_"):
        background = background + 1
    else:
        fruit = fruit + 1
print("รูป Kaggle ที่ตีกรอบแล้ว", fruit, "รูป (ควรได้ 3000)")
print("รูปพื้นหลังที่ไม่มีผลไม้", background, "รูป (ควรได้ 126)")
if len(images) != len(labels):
    raise Exception("รูปกับไฟล์ป้ายไม่เท่ากัน (" + str(len(images)) + " กับ " + str(len(labels)) + ") ให้สร้าง KaggleBoxes.zip ใหม่")
if background == 0:
    print("!!! ไม่มีรูปพื้นหลัง แปลว่า KaggleBoxes.zip เป็นรุ่นเก่า ให้อัปโหลดตัวใหม่")

## 5) เตรียมข้อมูล
แบ่งชุดเหมือนรอบ 1 แล้วทำ `data_more.yaml` เพิ่ม = ชุดเทรนเดิม + รูป Kaggle + รูปพื้นหลัง (val และ test ใช้ชุดเดิม)

In [ ]:
!python "{CODE}/prepare_data.py" --src "{SRC}" --out /content/data_5fruits --extra "{EXTRA}"

# เช็กผล : ต้องมีทั้ง data.yaml และ data_more.yaml
for f in ["data.yaml", "data_more.yaml"]:
    if not os.path.exists("/content/data_5fruits/" + f):
        raise Exception("ไม่มี " + f + " เตรียมข้อมูลไม่สำเร็จ ดูข้อความ error ด้านบน")
for part in ["train", "val", "test"]:
    print(part, len(os.listdir("/content/data_5fruits/" + part + "/images")), "รูป")
print(open("/content/data_5fruits/data_more.yaml", encoding="utf-8").read())
!cp /content/data_5fruits/dataset_counts.csv "{OUT}/"

## 6) เริ่มเทรน 4 การทดลอง
ลำดับ : 5_cosine_lr → 8_cosine_more_data → 4_rmsprop → 9_rmsprop_low_lr

In [ ]:
!python "{CODE}/train_yolo.py" --data /content/data_5fruits/data.yaml --more-data /content/data_5fruits/data_more.yaml --only 5_cosine_lr,8_cosine_more_data,4_rmsprop,9_rmsprop_low_lr --out "{OUT}" --external "{EXTERNAL}"

## 7) ดูผล

In [ ]:
import glob
from IPython.display import Image, Markdown, display

display(Markdown(open(OUT + "/results/summary.md", encoding="utf-8").read()))
print(open(OUT + "/models/class_info.json", encoding="utf-8").read())

for name in ["compare_more_data", "compare_rmsprop_lr", "compare_all", "confusion_matrix_normalized_best"]:
    file = OUT + "/results/" + name + ".png"
    if os.path.exists(file):
        print(name)
        display(Image(file))

print(open(OUT + "/results/per_class_best.csv", encoding="utf-8").read())

# ผลทดสอบกับรูปจากแหล่งอื่น ของทุกการทดลอง
for f in sorted(glob.glob(OUT + "/results/external_*_summary.txt")):
    print(open(f, encoding="utf-8").read())

## 8) เช็กว่ายังตีกรอบหน้าคนเป็นผลไม้ไหม
ใช้รูปคน (zidane.jpg) กับรูปรถบัส (bus.jpg) ที่มากับ ultralytics ไม่มีผลไม้ในรูป ค่าที่ดีคือไม่เจอกรอบเลย
(ความมั่นใจขั้นต่ำ 0.40 เท่ากับที่เว็บใช้)

In [ ]:
from ultralytics.utils import ASSETS

for name in ["5_cosine_lr", "8_cosine_more_data"]:
    best = OUT + "/runs/" + name + "/weights/best.pt"
    if not os.path.exists(best):
        continue
    model = ultralytics.YOLO(best)
    for picture in ["zidane.jpg", "bus.jpg"]:
        r = model.predict(str(ASSETS / picture), conf=0.40, verbose=False)[0]
        found = []
        for i in range(len(r.boxes)):
            found.append(r.names[int(r.boxes.cls[i])] + " " + str(round(float(r.boxes.conf[i]) * 100)) + "%")
        if len(found) == 0:
            print(name, "|", picture, "-> ไม่เจอผลไม้ (ถูกต้อง)")
        else:
            print(name, "|", picture, "-> ตีกรอบผิด:", found)

## 9) ลองให้โมเดลที่ดีที่สุดทายรูปใน test

In [ ]:
model = ultralytics.YOLO(OUT + "/models/best_detect.pt")
test_images = sorted(glob.glob("/content/data_5fruits/test/images/*.jpg"))[:4]
model.predict(test_images, conf=0.40, save=True, project="/content/demo", name="pred", exist_ok=True, verbose=False)
for f in sorted(glob.glob("/content/demo/pred/*.jpg")):
    display(Image(f, width=500))

## 10) เอาไปใช้ต่อ
- ส่งลิงก์โน้ตบุ๊กนี้ (ที่รันเสร็จแล้ว) ให้ทีมตรวจ เพื่อเลือกโมเดลใส่เว็บ
  ดู 3 อย่าง : Test F1 / ผลทดสอบรูปจากแหล่งอื่น (external) / ตีกรอบหน้าคนหรือไม่ (ข้อ 8)
- `KhadSan_detect_v2/results/` → วางใน `results/round2/` ของโปรเจกต์
- ไม่ต้องโหลด `runs/` (ไฟล์ใหญ่ เก็บไว้ใน Drive พอ)